In [1]:
import pandas as pd

# 1. ส่วนกำหนดข้อมูล (สามารถ เพิ่ม/ลบ/แก้ไข รายการสินค้าตรงนี้ได้เลย)
# ข้อมูลจำลองตามตารางที่ 5.2 ในภาพ
products_data = [
    {"product": "A", "sales": 1000000, "price": 2.50},
    {"product": "B", "sales": 250000, "price": 0.55},
    {"product": "C", "sales": 150000, "price": 6.50},
    {"product": "D", "sales": 300000, "price": 1.00},
    {"product": "E", "sales": 100000, "price": 1.50},
    {"product": "F", "sales": 700000, "price": 1.43},
    {"product": "G", "sales": 500000, "price": 9.00},
    {"product": "H", "sales": 15000, "price": 4.98},
    {"product": "J", "sales": 1000000, "price": 0.75},
    {"product": "K", "sales": 600000, "price": 1.62},
    {"product": "L", "sales": 25000, "price": 2.50},
    {"product": "M", "sales": 4200, "price": 15.00},
    {"product": "N", "sales": 1000000, "price": 5.00},
    {"product": "O", "sales": 2850000, "price": 10.00},
    {"product": "P", "sales": 10000, "price": 0.83},
    {"product": "Q", "sales": 355000, "price": 0.99},
    {"product": "R", "sales": 40000, "price": 1.37},
    {"product": "S", "sales": 393000, "price": 1.85},
    {"product": "T", "sales": 250000, "price": 4.12}, 
]

# แปลงข้อมูลเป็น DataFrame
df = pd.DataFrame(products_data)

# 2. ขั้นตอนการคำนวณ
# 2.1 คำนวณมูลค่ารวม (ยอดขาย x ราคา)
df['total_value'] = df['sales'] * df['price']

# 2.2 เรียงลำดับจากมูลค่ามากไปน้อย (สำคัญมากสำหรับ ABC Analysis)
df = df.sort_values(by='total_value', ascending=False).reset_index(drop=True)

# 2.3 คำนวณมูลค่าสะสม (Cumulative Value)
df['accumulated_value'] = df['total_value'].cumsum()

# 2.4 คำนวณ % สะสม
total_sum = df['total_value'].sum()
df['percent_accumulate'] = (df['accumulated_value'] / total_sum) * 100

# 3. การจัดกลุ่ม ABC (ตามเกณฑ์ 80-20 หรือตามภาพที่ 5.3)
# เกณฑ์: A <= 80%, B <= 95%, C > 95% (ปรับแก้ตัวเลขได้ตามต้องการ)
def assign_group(pct):
    # หมายเหตุ: ใช้ 80.1 เพื่อให้ครอบคลุมกรณีที่ตัวเลขเกิน 80 มานิดหน่อยเหมือนในตัวอย่าง (เช่น 80.05)
    if pct <= 80.1: 
        return 'A'
    elif pct <= 95.5:
        return 'B'
    else:
        return 'C'

df['group'] = df['percent_accumulate'].apply(assign_group)

# 4. จัดรูปแบบการแสดงผลให้สวยงามเหมือนตารางที่ 5.3
# เปลี่ยนชื่อคอลัมน์ภาษาไทย
output_df = df.rename(columns={
    'product': 'ผลิตภัณฑ์',
    'sales': 'ยอดขาย (ชิ้น)',
    'price': 'ราคา (บาท/ชิ้น)',
    'total_value': 'มูลค่ารวม',
    'percent_accumulate': '% สะสม',
    'group': 'กลุ่ม'
})

# จัดรูปแบบตัวเลข (ใส่ลูกน้ำและทศนิยม)
pd.options.display.float_format = '{:,.2f}'.format

# แสดงผลลัพธ์
display(output_df)

# สรุปยอดตามกลุ่ม
print("\n--- สรุปข้อมูลตามกลุ่ม ---")
summary = output_df.groupby('กลุ่ม')['มูลค่ารวม'].agg(['count', 'sum'])
summary['% ของมูลค่าทั้งหมด'] = (summary['sum'] / total_sum) * 100
display(summary)

,ผลิตภัณฑ์,ยอดขาย (ชิ้น),ราคา (บาท/ชิ้น),มูลค่ารวม,accumulated_value,% สะสม,กลุ่ม
0,O,2850000,10.00,"28,500,000.00","28,500,000.00",60.44,A
1,N,1000000,5.00,"5,000,000.00","33,500,000.00",71.04,A
2,G,500000,9.00,"4,500,000.00","38,000,000.00",80.58,B
3,A,1000000,2.50,"2,500,000.00","40,500,000.00",85.88,B
4,T,250000,4.12,"1,030,000.00","41,530,000.00",88.07,B
5,F,700000,1.43,"1,001,000.00","42,531,000.00",90.19,B
6,C,150000,6.50,"975,000.00","43,506,000.00",92.26,B
7,K,600000,1.62,"972,000.00","44,478,000.00",94.32,B
8,J,1000000,0.75,"750,000.00","45,228,000.00",95.91,C
9,S,393000,1.85,"727,050.00","45,955,050.00",97.45,C



--- สรุปข้อมูลตามกลุ่ม ---


,count,sum,% ของมูลค่าทั้งหมด
กลุ่ม,,,
A,2,"33,500,000.00",71.04
B,6,"10,978,000.00",23.28
C,11,"2,679,300.00",5.68
